# Système RAG YARA — Génération de Règles YARA

**Domaine:** Cybersécurité  
**Dataset:** ~30K règles YARA validées (yara-python)  
**Architectures RAG:** Hybride | Rerank | Agentic  
**LLMs:** Phi-2 | TinyLlama | Mistral-7B

---
### Pipeline
User décrit malware → RAG Hybride (BM25+FAISS) → LLM génère règle YARA

## Cellule 1 — Installation des dépendances

In [1]:
# Exécuter une seule fois
import os
if not os.path.exists('.deps_installed'):
    import subprocess
    subprocess.run(['pip', 'install', '-q',
        'sentence-transformers',
        'faiss-cpu',
        'rank-bm25',
        'transformers',
        'torch',
        'gradio',
        'accelerate',
        'bitsandbytes',
        'einops',
        'scikit-learn',
        'pandas',
        'numpy'
    ])
    open('.deps_installed', 'w').close()
    print('Dépendances installées.')
else:
    print('Dépendances déjà installées — skip.')

Dépendances installées.


## Cellule 2 — Imports

In [2]:
import os, re, json, time, pickle, warnings
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
import torch
import gradio as gr



print('Imports OK')
print(f'GPU disponible: {torch.cuda.is_available()}')

Imports OK
GPU disponible: True


In [3]:
# Paths
DATASET_PATH    = '/content/yara_step2.csv'
EMBED_MODEL_DIR = '/content/embedding_model'
FAISS_PATH      = '/content/yara_faiss.index'
BM25_PATH       = '/content/yara_bm25.pkl'
DOCS_PATH       = '/content/yara_docs.pkl'
EMBED_PATH      = '/content/yara_embeddings.pkl'

## Cellule 3 — Chargement et préparation du dataset

In [4]:
print('Chargement du dataset...')
df_raw = pd.read_csv(DATASET_PATH, dtype=str).fillna('')
print(f'  Total chargé : {len(df_raw):,} règles')

# Garder uniquement les règles valides (déjà vérifiées par yara-python)
df = df_raw[df_raw['is_valid'].str.lower().isin(['true', '1', 'yes'])].copy()
print(f'  Après filtre is_valid=True : {len(df):,} règles')

# Normaliser les colonnes
# Support rule_text ou rule_body
if 'rule_body' not in df.columns and 'rule_text' in df.columns:
    df.rename(columns={'rule_text': 'rule_body'}, inplace=True)
if 'rule_body' not in df.columns and 'rule' in df.columns:
    df.rename(columns={'rule': 'rule_body'}, inplace=True)

# Utiliser predicted_type si disponible, sinon malware_type
if 'predicted_type' in df.columns:
    df['type'] = df['predicted_type']
elif 'malware_type' in df.columns:
    df['type'] = df['malware_type']
else:
    df['type'] = 'malware'

# Fix 2 — Texte enrichi pour embedding (rule_name + type + description)
# Donne 3x plus de signal au modèle d'embedding
df['embed_text'] = (
    df['rule_name'].str.replace('_', ' ') + ' ' +
    df['type'] + ' ' +
    df['description']
).str.strip()

# Source URL pour bouton référence
if 'source_url' not in df.columns:
    df['source_url'] = ''

df = df.reset_index(drop=True)
documents = df.to_dict('records')

print(f'  Documents prêts : {len(documents):,}')
print(f'  Colonnes        : {list(df.columns)}')
print(f'\n  Distribution des types :')
print(df['type'].value_counts().head(10).to_string())
print(f'\n  Exemple embed_text :')
print(f'  {df["embed_text"].iloc[0][:120]}')

Chargement du dataset...
  Total chargé : 30,924 règles
  Après filtre is_valid=True : 30,924 règles
  Documents prêts : 30,924
  Colonnes        : ['id', 'rule_name', 'malware_type', 'description', 'rule_body', 'author', 'date', 'tags', 'source_url', 'source_name', 'source_type', 'filename', 'content_hash', 'is_valid', 'enrichment_level', 'original_type', 'predicted_type', 'type', 'embed_text']

  Distribution des types :
type
malware        15842
rat             3628
apt             3261
ransomware      1584
hacktool        1394
backdoor         844
exploit          680
loader           547
webshell         531
infostealer      526

  Exemple embed_text :
  MachO malware Detect Mach-O file produced by pyinstaller


## Cellule 4 — Modèle d'embedding (téléchargé une seule fois)

In [5]:
# Fix 1 — Modèle orienté code/cybersécurité
# Remplace all-MiniLM-L6-v2 qui ne comprend pas les hex patterns YARA
EMBED_MODEL_NAME = 'BAAI/bge-base-en-v1.5'

if os.path.exists(EMBED_MODEL_DIR):
    print(f'Modèle embedding déjà téléchargé — chargement depuis {EMBED_MODEL_DIR}...')
    embedding_model = SentenceTransformer(EMBED_MODEL_DIR)
else:
    print(f'Téléchargement du modèle embedding : {EMBED_MODEL_NAME}...')
    embedding_model = SentenceTransformer(EMBED_MODEL_NAME, trust_remote_code=True)
    embedding_model.save(EMBED_MODEL_DIR)
    print(f'Modèle sauvegardé → {EMBED_MODEL_DIR}')

EMBED_DIM = embedding_model.get_sentence_embedding_dimension()
print(f'Modèle prêt — dimension : {EMBED_DIM}')

Téléchargement du modèle embedding : BAAI/bge-base-en-v1.5...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle sauvegardé → /content/embedding_model
Modèle prêt — dimension : 768


## Cellule 5 — Indexation FAISS + BM25 (calculée une seule fois)

In [6]:
embed_texts = df['embed_text'].tolist()

if os.path.exists(FAISS_PATH) and os.path.exists(BM25_PATH) and os.path.exists(DOCS_PATH):
    print('Index déjà calculés — chargement...')
    faiss_index = faiss.read_index(FAISS_PATH)
    with open(BM25_PATH, 'rb') as f:
        bm25 = pickle.load(f)
    with open(DOCS_PATH, 'rb') as f:
        documents = pickle.load(f)
    with open(EMBED_PATH, 'rb') as f:
        doc_embeddings = pickle.load(f)
    print(f'  FAISS : {faiss_index.ntotal:,} vecteurs')
    print(f'  BM25  : {len(documents):,} documents')
else:
    print(f'Calcul des embeddings pour {len(embed_texts):,} règles...')
    doc_embeddings = embedding_model.encode(
        embed_texts,
        show_progress_bar=True,
        convert_to_numpy=True,
        batch_size=64
    )
    print(f'  Shape embeddings : {doc_embeddings.shape}')

    # Index FAISS
    print('Construction index FAISS...')
    faiss_index = faiss.IndexFlatIP(EMBED_DIM)  # Inner Product (cosine après normalisation)
    norms = np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    normalized = doc_embeddings / (norms + 1e-10)
    faiss_index.add(normalized.astype('float32'))

    # Index BM25
    print('Construction index BM25...')
    tokenized = [t.lower().split() for t in embed_texts]
    bm25 = BM25Okapi(tokenized)

    # Sauvegardes
    faiss.write_index(faiss_index, FAISS_PATH)
    with open(BM25_PATH, 'wb') as f:
        pickle.dump(bm25, f)
    with open(DOCS_PATH, 'wb') as f:
        pickle.dump(documents, f)
    with open(EMBED_PATH, 'wb') as f:
        pickle.dump(doc_embeddings, f)

    print(f'Index sauvegardés → {FAISS_PATH}, {BM25_PATH}')

print('Indexation terminée.')

Calcul des embeddings pour 30,924 règles...


Batches:   0%|          | 0/484 [00:00<?, ?it/s]

  Shape embeddings : (30924, 768)
Construction index FAISS...
Construction index BM25...
Index sauvegardés → /content/yara_faiss.index, /content/yara_bm25.pkl
Indexation terminée.


## Cellule 6 — Modèle LLM (téléchargé une seule fois)

In [7]:
# ================================================================
# 3 LLMs disponibles — décommenter celui à utiliser
# Un seul actif à la fois pour économiser l'espace
# ================================================================

# Option 1 — Phi-2 (2.7B) — ACTIF pour testing
# Léger, performant en code, bon pour YARA
#LLM_NAME = 'microsoft/phi-2'
#LLM_DIR  = '/content/llm_phi2'

# Option 2 — TinyLlama (1.1B) — le plus léger
#LLM_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
#LLM_DIR  = '/content/llm_tinyllama'

 #Option 3 — Mistral-7B — meilleure qualité
LLM_NAME = 'mistralai/Mistral-7B-Instruct-v0.1'
LLM_DIR  = '/content/llm_mistral'

# ================================================================

USE_4BIT = True

if os.path.exists(LLM_DIR):
    print(f'LLM déjà téléchargé — chargement depuis {LLM_DIR}...')
    tokenizer = AutoTokenizer.from_pretrained(LLM_DIR)
    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True
        )
        llm = AutoModelForCausalLM.from_pretrained(
            LLM_DIR,
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True
        )
    else:
        llm = AutoModelForCausalLM.from_pretrained(
            LLM_DIR,
            device_map='cpu',
            trust_remote_code=True,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True
        )
else:
    print(f'Téléchargement LLM : {LLM_NAME}...')
    tokenizer = AutoTokenizer.from_pretrained(LLM_NAME, trust_remote_code=True)
    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True
        )
        llm = AutoModelForCausalLM.from_pretrained(
            LLM_NAME,
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True
        )
    else:
        llm = AutoModelForCausalLM.from_pretrained(
            LLM_NAME,
            device_map='cpu',
            trust_remote_code=True,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True
        )
    tokenizer.save_pretrained(LLM_DIR)
    llm.save_pretrained(LLM_DIR)
    print(f'LLM sauvegardé → {LLM_DIR}')

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm_pipeline = pipeline(
    'text-generation',
    model=llm,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)

ACTIVE_LLM = LLM_NAME.split('/')[-1]
print(f'LLM actif : {ACTIVE_LLM}')

def generate(prompt: str, max_tokens: int = 512) -> str:
    try:
        out = llm_pipeline(prompt, max_new_tokens=max_tokens)
        return out[0]['generated_text'].strip()
    except Exception as e:
        return f'Erreur génération: {e}'

Téléchargement LLM : mistralai/Mistral-7B-Instruct-v0.1...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM sauvegardé → /content/llm_mistral
LLM actif : Mistral-7B-Instruct-v0.1


In [23]:
def generate(prompt: str, max_tokens: int = 400) -> str:
    try:
        out = llm_pipeline(prompt, max_new_tokens=max_tokens, return_full_text=False)
        text = out[0]['generated_text'].strip()

        # 1. Supprimer les préfixes numériques comme "1 {"
        if re.match(r'^\d+\s*\{', text):
            text = re.sub(r'^\d+\s*\{', 'rule Detect_Malware {', text)

        # 2. Si la règle ne commence pas par "rule ", l'ajouter
        if not re.search(r'^rule\s+\w+', text, re.IGNORECASE):
            text = 'rule Detect_Malware {\n' + text

        # 3. Corriger les sections mal formatées : meta { -> meta: , etc.
        text = re.sub(r'meta\s*\{', 'meta:', text, flags=re.IGNORECASE)
        text = re.sub(r'strings\s*\{', 'strings:', text, flags=re.IGNORECASE)
        text = re.sub(r'condition\s*\{', 'condition:', text, flags=re.IGNORECASE)

        # 4. Supprimer les lignes de date isolée (ex: "2025-01-01 {")
        text = re.sub(r'\d{4}-\d{2}-\d{2}\s*\{', '', text)

        # 5. S'assurer que la règle se termine par une accolade fermante
        open_braces = text.count('{')
        close_braces = text.count('}')
        if open_braces > close_braces:
            text += '}' * (open_braces - close_braces)

        # 6. Nettoyer les noms de règle invalides (ex: "rule 1 {" -> "rule Detect_Malware {")
        text = re.sub(r'rule\s+\d+\s*\{', 'rule Detect_Malware {', text)

        return text.strip()
    except Exception as e:
        return f'Erreur génération: {e}'

## Cellule 7 — Re-ranker (téléchargé une seule fois)

In [8]:
RERANKER_DIR  = '/content/reranker_model'
RERANKER_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

if os.path.exists(RERANKER_DIR):
    print('Re-ranker déjà téléchargé — chargement...')
    reranker = CrossEncoder(RERANKER_DIR)
else:
    print(f'Téléchargement re-ranker : {RERANKER_NAME}...')
    reranker = CrossEncoder(RERANKER_NAME)
    reranker.save(RERANKER_DIR)
    print(f'Re-ranker sauvegardé → {RERANKER_DIR}')

print('Re-ranker prêt.')

Téléchargement re-ranker : cross-encoder/ms-marco-MiniLM-L-6-v2...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Re-ranker sauvegardé → /content/reranker_model
Re-ranker prêt.


## Cellule 8 — Fonctions de retrieval

In [41]:
def retrieve_hybrid(query: str, k: int = 5, alpha: float = 0.6) -> List[Dict]:
    """
    RAG Hybride : BM25 (sparse) + FAISS (dense) fusionnés.
    alpha=0.6 → favorise légèrement FAISS (sémantique)
    BM25 capture les strings YARA exacts (hex, API names)
    FAISS capture le sens général de la requête
    """
    # Dense — FAISS
    q_emb = embedding_model.encode([query], convert_to_numpy=True)
    q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-10)
    scores_faiss, indices_faiss = faiss_index.search(q_norm.astype('float32'), k * 3)
    dense = {int(idx): float(score) for idx, score in zip(indices_faiss[0], scores_faiss[0])}

    # Sparse — BM25
    tokens = query.lower().split()
    bm25_scores = bm25.get_scores(tokens)
    top_bm25 = np.argsort(bm25_scores)[::-1][:k * 3]
    max_bm25 = bm25_scores[top_bm25[0]] if bm25_scores[top_bm25[0]] > 0 else 1
    sparse = {int(i): float(bm25_scores[i]) / max_bm25 for i in top_bm25}

    # Fusion
    max_dense = max(dense.values()) if dense else 1
    all_ids = set(dense.keys()) | set(sparse.keys())
    combined = {
        idx: alpha * (dense.get(idx, 0) / max_dense) + (1 - alpha) * sparse.get(idx, 0)
        for idx in all_ids
    }

    top_ids = sorted(combined, key=combined.get, reverse=True)[:k]
    results = []
    for idx in top_ids:
        doc = documents[idx].copy()
        doc['retrieval_score'] = round(combined[idx], 4)
        doc['dense_score']     = round(dense.get(idx, 0), 4)
        doc['sparse_score']    = round(sparse.get(idx, 0), 4)
        results.append(doc)
    return results


def retrieve_with_rerank(query: str, k_candidates: int = 15, k_final: int = 5) -> List[Dict]:
    """
    RAG Rerank : Hybride → CrossEncoder pour re-classer.
    Récupère 15 candidats, re-classe, garde top 5.
    """
    candidates = retrieve_hybrid(query, k=k_candidates)
    pairs = [[query, doc['embed_text']] for doc in candidates]
    rerank_scores = reranker.predict(pairs)
    for doc, score in zip(candidates, rerank_scores):
        doc['rerank_score'] = float(score)
    return sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)[:k_final]


print('Fonctions retrieval définies.')

# Test rapide
test = retrieve_hybrid('ransomware encrypt files AES', k=3)
print(f'Test retrieval hybride — top 3 :')
for d in test:
    print(f"  [{d['type']}] {d['rule_name']} (score: {d['retrieval_score']})")

Fonctions retrieval définies.
Test retrieval hybride — top 3 :
  [ransomware] BINARYALERT_Ransomware_Windows_Zcrypt (score: 0.8492)
  [ransomware] DecryptMyFiles (score: 0.6)
  [ransomware] GetCrypt (score: 0.5833)


## Cellule 9 — Prompt builder et 3 architectures RAG

In [78]:
def build_context(docs: List[Dict], n: int = 2) -> str:
    """Construit le contexte syntaxique à partir des règles récupérées.
    On envoie uniquement le squelette de la règle pour guider la syntaxe,
    pas la règle entière (évite la copie).
    """
    parts = []
    for i, doc in enumerate(docs[:n], 1):
        rule_body = str(doc.get('rule_body', ''))
        # Extraire uniquement strings + condition (pas meta)
        strings_match   = re.search(r'strings\s*:(.*?)(?=condition|\})', rule_body, re.DOTALL | re.IGNORECASE)
        condition_match = re.search(r'condition\s*:(.*?)(?=\}|$)',        rule_body, re.DOTALL | re.IGNORECASE)
        strings_part   = strings_match.group(1).strip()[:300]   if strings_match   else ''
        condition_part = condition_match.group(1).strip()[:150]  if condition_match else ''
        parts.append(
            f"// Example {i} [{doc.get('type','unknown')}]\n"
            f"// strings:\n{strings_part}\n"
            f"// condition: {condition_part}"
        )
    return '\n\n'.join(parts)


# Version corrigée
def build_prompt(query, context, ref_url='', ref_desc='', rule_name='Detect_Malware'):
    ref_line = f'reference = "{ref_url[:80]}"' if ref_url else ''
    return f"""<s>[INST] Generate a YARA rule that detects: {query}

Examples (do not copy, only inspire):
{context}

You MUST follow this template exactly, replace CAPITALIZED text:

rule {rule_name} {{
    meta:
        author = "RAG-System"
        description = "A short description of what the rule detects"
        date = "2025-01-01"
        {ref_line}
    strings:
        $a = "first string" ascii
        $b = "second string" ascii
    condition:
        $a and $b
}}

CONSTRAINTS:
- Every string must be properly quoted (") or hex-bracketed {{ }}.
- Hex strings must be closed with }}.
- The condition must only refer to strings that exist.
- No extra text, no markdown, no comments outside the rule.
- Output ONLY the rule.

Now write the rule: [/INST]
rule {rule_name} {{
    meta:
        author = "RAG-System"
        description = \""""

# ── Architecture 1 : RAG Hybride ─────────────────────────────────────────
def rag_hybrid(query: str) -> Tuple[str, List[Dict]]:
    docs      = retrieve_hybrid(query, k=5)
    ref_url   = docs[0].get('source_url',  '') if docs else ''
    ref_desc  = docs[0].get('description', '') if docs else ''
    rule_name = docs[0].get('rule_name', 'Detect_Malware') if docs else 'Detect_Malware'
    # Nettoyer rule_name — enlever chiffres au début
    if rule_name and rule_name[0].isdigit():
        rule_name = 'Rule_' + rule_name
    context   = build_context(docs, n=2)
    prompt    = build_prompt(query, context, ref_url, ref_desc, rule_name)
    rule      = generate(prompt)
    return rule, docs


# ── Architecture 2 : RAG Rerank ──────────────────────────────────────────
def rag_rerank(query: str) -> Tuple[str, List[Dict]]:
    docs      = retrieve_with_rerank(query, k_candidates=15, k_final=5)
    ref_url   = docs[0].get('source_url',  '') if docs else ''
    ref_desc  = docs[0].get('description', '') if docs else ''
    rule_name = docs[0].get('rule_name', 'Detect_Malware') if docs else 'Detect_Malware'
    if rule_name and rule_name[0].isdigit():
        rule_name = 'Rule_' + rule_name
    context   = build_context(docs, n=2)
    prompt    = build_prompt(query, context, ref_url, ref_desc, rule_name)
    rule      = generate(prompt)
    return rule, docs


# ── Architecture 3 : Agentic RAG ─────────────────────────────────────────
import yara

def is_valid_yara(rule_text: str) -> bool:
    try:
        yara.compile(source=rule_text)
        return True
    except Exception as e:
        # Optionnel : print(e) pour debug
        return False


def rag_agentic(query: str, max_iterations: int = 3) -> Tuple[str, List[Dict]]:
    docs      = retrieve_with_rerank(query, k_candidates=15, k_final=5)
    ref_url   = docs[0].get('source_url',  '') if docs else ''
    ref_desc  = docs[0].get('description', '') if docs else ''
    rule_name = docs[0].get('rule_name', 'Detect_Malware') if docs else 'Detect_Malware'
    if rule_name and rule_name[0].isdigit():
        rule_name = 'Rule_' + rule_name
    context   = build_context(docs, n=2)
    prompt    = build_prompt(query, context, ref_url, ref_desc, rule_name)
    rule      = generate(prompt)
    for iteration in range(max_iterations):
        if is_valid_yara(rule):
            break
        print(f'  [Agentic] Iteration {iteration+1} — correction...')
        fix_prompt = build_prompt(query, context, ref_url, ref_desc, rule_name)
        rule = generate(fix_prompt)
    return rule, docs


# Mapping pour l'interface
RAG_FUNCTIONS = {
    'RAG Hybride': rag_hybrid,
    'RAG Rerank' : rag_rerank,
    'Agentic RAG': rag_agentic
}

print('Architectures RAG définies : Hybride | Rerank | Agentic')


Architectures RAG définies : Hybride | Rerank | Agentic


In [79]:
def generate(prompt: str, max_tokens: int = 400) -> str:
    try:
        out = llm_pipeline(prompt, max_new_tokens=max_tokens, return_full_text=False)
        text = out[0]['generated_text'].strip()

        # 1. Supprimer les préfixes numériques
        text = re.sub(r'^\d+\s*\{', 'rule Detect_Malware {', text)

        # 2. Ajouter "rule " si absent
        if not re.search(r'^rule\s+\w+', text, re.IGNORECASE):
            match = re.search(r'^([A-Za-z_][A-Za-z0-9_]*)\s*\{', text, re.MULTILINE)
            if match:
                rule_name = match.group(1)
                text = re.sub(rf'^{re.escape(rule_name)}\s*{{', f'rule {rule_name} {{', text, flags=re.MULTILINE)
            else:
                text = 'rule Detect_Malware {\n' + text

        # 3. Supprimer le faux "rule Detect_Malware {" suivi d'un autre rule
        text = re.sub(r'rule\s+Detect_Malware\s*\{\s*\n\s*(?=\w+\s*\{)', '', text, flags=re.IGNORECASE)

        # 4. Corriger les noms commençant par un chiffre
        match = re.search(r'rule\s+([0-9][A-Za-z0-9_]*)\s*\{', text)
        if match:
            text = text.replace(f'rule {match.group(1)}', f'rule Rule_{match.group(1)}')

        # 5. Supprimer les appels à string.find() et autres fonctions Python
        lines = text.split('\n')
        text = '\n'.join([l for l in lines if 'string.find(' not in l and '.find(' not in l and 'import ' not in l and 'def ' not in l])

        # 6. Nettoyer les strings hex : enlever les commentaires // et tout ce qui suit sur la ligne
        def clean_hex_line(m):
            content = m.group(1)
            # Supprimer les commentaires // et tout texte après
            content = re.sub(r'\s*//.*', '', content)
            return f'{{ {content.strip()} }}'
        text = re.sub(r'\{\s*([^}]*?)\s*\}', clean_hex_line, text, flags=re.DOTALL)

        # 7. Déduplication des strings (par valeur)
        lines = text.split('\n')
        new_lines = []
        seen_values = set()
        in_strings = False
        for line in lines:
            if re.search(r'^\s*strings\s*[:{]', line, re.IGNORECASE):
                in_strings = True
                new_lines.append(line)
                continue
            if in_strings and re.search(r'^\s*condition\s*[:{]', line, re.IGNORECASE):
                in_strings = False
            if in_strings:
                m = re.match(r'\s*(\$\w+)\s*=\s*({[^}]+}|"[^"]+")(?:\s+(?:ascii|wide|fullword))*', line)
                if m:
                    name, val = m.groups()
                    if val in seen_values:
                        continue
                    seen_values.add(val)
            new_lines.append(line)
        text = '\n'.join(new_lines)

        # 8. Correction de la section meta : si "meta:" manque mais que des champs sont présents
        if re.search(r'rule\s+\w+\s*\{\s*(?!meta:)', text, re.IGNORECASE):
            # Insérer "meta:" après l'accolade ouvrante
            text = re.sub(r'(rule\s+\w+\s*\{\s*)', r'\1meta:\n    author = "RAG-System"\n', text, flags=re.IGNORECASE)

        # 9. Doubles accolades
        text = text.replace('{ {', '{').replace('}}', '}')

        # 10. Correction des sections mal formatées
        text = re.sub(r'meta\s*\{', 'meta:', text, flags=re.IGNORECASE)
        text = re.sub(r'strings\s*\{', 'strings:', text, flags=re.IGNORECASE)
        text = re.sub(r'condition\s*\{', 'condition:', text, flags=re.IGNORECASE)

        # 11. Nettoyer la condition : supprimer les références à des strings inexistantes
        defined_strings = set(re.findall(r'\$(\w+)\s*=', text))
        def clean_condition(m):
            cond = m.group(1)
            for s in re.findall(r'\$(\w+)', cond):
                if s not in defined_strings:
                    cond = re.sub(rf'\${re.escape(s)}\b', '', cond)
            cond = re.sub(r'\s+and\s+and\s+', ' and ', cond)
            cond = re.sub(r'\s+or\s+or\s+', ' or ', cond)
            cond = re.sub(r'\s+and\s*$', '', cond)
            cond = re.sub(r'^\s*and\s+', '', cond)
            if not cond.strip():
                cond = 'true'
            return f'condition: {cond}'
        text = re.sub(r'condition\s*:\s*(.*?)(?=\n\S|\Z)', clean_condition, text, flags=re.DOTALL)

        # 12. Fermer les accolades manquantes
        open_b = text.count('{')
        close_b = text.count('}')
        if open_b > close_b:
            text += '}' * (open_b - close_b)

        return text.strip()
    except Exception as e:
        return f'Erreur génération: {e}'

## Cellule 10 — Métriques d'évaluation

In [80]:
def metric_syntax_valid(rule: str) -> bool:
    """Règle structurellement valide (meta + strings + condition)."""
    return is_valid_yara(rule)


def metric_similarity(generated: str, retrieved_docs: List[Dict]) -> float:
    """Similarité cosine entre règle générée et règles récupérées."""
    if not retrieved_docs:
        return 0.0
    gen_emb  = embedding_model.encode([generated], convert_to_numpy=True)
    ref_texts = [d.get('embed_text', '') for d in retrieved_docs]
    ref_embs  = embedding_model.encode(ref_texts, convert_to_numpy=True)
    sims = cosine_similarity(gen_emb, ref_embs)[0]
    return float(np.mean(sims))


def metric_hallucination(generated: str, retrieved_docs: List[Dict]) -> float:
    """
    Fix 4 — Métrique d'hallucination corrigée.
    Vérifie si le TYPE de malware généré correspond à la requête.
    NE pénalise PAS les nouveaux patterns hex (c'est le but du RAG).
    Pénalise uniquement les incohérences sémantiques.
    """
    if not retrieved_docs:
        return 1.0
    # Vérifier présence de la section meta (hallucination = pas de meta)
    has_meta   = bool(re.search(r'meta\s*:', generated, re.IGNORECASE))
    # Vérifier présence d'au moins une string définie
    has_string = bool(re.search(r'\$\w+\s*=', generated))
    # Vérifier que la condition référence les strings définies
    string_vars = re.findall(r'\$(\w+)\s*=', generated)
    condition_match = re.search(r'condition:(.*)', generated, re.DOTALL | re.IGNORECASE)
    condition_text  = condition_match.group(1) if condition_match else ''
    strings_used = any(f'${v}' in condition_text for v in string_vars) if string_vars else False

    score = 0
    if not has_meta:    score += 0.4
    if not has_string:  score += 0.3
    if string_vars and not strings_used: score += 0.3
    return min(1.0, score)


def metric_retrieval_relevance(retrieved_docs: List[Dict], query: str) -> float:
    """Proportion de documents récupérés avec description non-vide et type spécifique."""
    if not retrieved_docs:
        return 0.0
    relevant = sum(
        1 for d in retrieved_docs
        if len(str(d.get('description', ''))) > 30
        and str(d.get('type', 'malware')) != 'malware'
    )
    return relevant / len(retrieved_docs)


def evaluate_all_architectures(queries: List[str]) -> pd.DataFrame:
    """Évalue les 3 architectures sur une liste de requêtes."""
    results = []
    for arch_name, arch_func in RAG_FUNCTIONS.items():
        print(f'\n  Évaluation : {arch_name}')
        for query in queries:
            t0 = time.time()
            try:
                rule, docs = arch_func(query)
                latency   = time.time() - t0
                results.append({
                    'Architecture'         : arch_name,
                    'Query'                : query[:60],
                    'Syntaxe Valide'       : metric_syntax_valid(rule),
                    'Similarité'           : round(metric_similarity(rule, docs), 3),
                    'Hallucination'        : round(metric_hallucination(rule, docs), 3),
                    'Pertinence Retrieval' : round(metric_retrieval_relevance(docs, query), 3),
                    'Latence (s)'          : round(latency, 2)
                })
                print(f'    [{arch_name}] {query[:40]}... — {latency:.1f}s')
            except Exception as e:
                print(f'    ERREUR : {e}')
    df_res = pd.DataFrame(results)
    summary = df_res.groupby('Architecture').mean(numeric_only=True).round(3)
    summary['Score Global'] = (
        summary['Syntaxe Valide']       * 0.30 +
        summary['Similarité']           * 0.30 +
        (1 - summary['Hallucination'])  * 0.20 +
        summary['Pertinence Retrieval'] * 0.20
    ).round(3)
    print('\n=== RÉSULTATS ===')
    print(summary.sort_values('Score Global', ascending=False).to_string())
    return df_res


# Requêtes de test
TEST_QUERIES = [
    'Detect ransomware encrypting files with AES and requesting Bitcoin ransom',
    'Identify keylogger capturing keystrokes with GetAsyncKeyState',
    'Find rootkit hiding processes by hooking SSDT',
    'Detect backdoor opening reverse shell on port 4444',
    'Identify cryptominer using CPU resources to mine Monero'
]

print('Métriques définies.')
print(f'{len(TEST_QUERIES)} requêtes de test prêtes.')

Métriques définies.
5 requêtes de test prêtes.


## Cellule 11 — Lancer l'évaluation (optionnel)

In [66]:
# Décommenter pour lancer l'évaluation complète
# ATTENTION : prend ~10-20 min selon le LLM actif

eval_results = evaluate_all_architectures(TEST_QUERIES)
eval_results.to_csv('/content/rag_evaluation.csv', index=False)
print('Résultats sauvegardés → /content/rag_evaluation.csv')

print('Cellule évaluation prête — décommenter pour lancer.')


  Évaluation : RAG Hybride


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    [RAG Hybride] Detect ransomware encrypting files with ... — 35.7s


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    [RAG Hybride] Identify keylogger capturing keystrokes ... — 21.6s


...

  [Agentic] Iteration 1 — correction...


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Agentic] Iteration 2 — correction...


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [Agentic] Iteration 3 — correction...
    [Agentic RAG] Identify cryptominer using CPU resources... — 54.7s

=== RÉSULTATS ===
              Syntaxe Valide  Similarité  Hallucination  Pertinence Retrieval  Latence (s)  Score Global
Architecture                                                                                            
Agentic RAG              0.0       0.716           0.00                  0.80       61.220         0.575
RAG Rerank               0.0       0.715           0.06                  0.80       21.486         0.562
RAG Hybride              0.0       0.709           0.00                  0.72       19.634         0.557
Résultats sauvegardés → /content/rag_evaluation.csv
Cellule évaluation prête — décommenter pour lancer.


## Cellule 12 — Interface Gradio

In [81]:
# État partagé
last_response_state = {
    'rule'        : '',
    'description' : '',
    'reference'   : '',
    'docs'        : [],
    'valid'       : False,
    'latency'     : 0.0
}


def explain_rule(rule: str, query: str) -> str:
    if not rule or 'Erreur' in rule:
        return 'Aucune règle à expliquer.'

    # Extraire strings et condition
    strings = re.findall(r'(\$\w+\s*=\s*[^\n]+)', rule)
    strings_str = '\n'.join(strings[:6])
    condition_m = re.search(r'condition\s*:(.*?)(?=\}|$)', rule, re.DOTALL)
    condition_str = condition_m.group(1).strip()[:150] if condition_m else ''

    prompt = f"""<s>[INST] You are a cybersecurity analyst.
Explain this YARA rule line by line in simple English:

{rule[:800]}

Your explanation must cover:
1. One sentence: what malware/behavior this rule detects
2. Each string — what it looks for and why it matters for detection
3. The condition — what logic triggers the rule

Be concise and technical. No markdown. [/INST]
This rule detects"""

    try:
        out    = llm_pipeline(prompt, max_new_tokens=250, return_full_text=False)
        result = out[0]['generated_text'].strip()
        for stop in ['rule ', '[INST]', '```', 'Example']:
            if stop in result:
                result = result.split(stop)[0]
        return 'This rule detects ' + result.strip()
    except Exception as e:
        return f'Erreur explication: {e}'


def extract_meta_field(rule: str, field: str) -> str:
    """Extrait un champ de la section meta de la règle YARA."""
    pattern = rf'{field}\s*=\s*["\']?([^"\'\n]+)["\']?'
    match   = re.search(pattern, rule, re.IGNORECASE)
    return match.group(1).strip() if match else ''


def clean_rule_output(raw: str) -> str:
    """Extrait et corrige la règle YARA depuis la sortie brute."""
    # Supprimer tout ce qui précède la première occurrence de 'rule '
    if 'rule ' in raw.lower():
        raw = raw[raw.lower().index('rule '):]
    # Couper à la première occurrence de '```' ou d'un double saut de ligne après la règle
    for stop in ['```', '\n\n\n']:
        if stop in raw:
            raw = raw.split(stop)[0]
    # Nettoyer les artefacts
    raw = re.sub(r'^\d+\s*', '', raw)          # enlève '1 ' en début
    raw = re.sub(r'rule\s+\d+\s*\{', 'rule Detect_Malware {', raw)
    return raw.strip()


def generate_rule(query: str, rag_type: str, llm_choice: str) -> str:
    """
    Fonction principale appelée par Gradio au clic sur Générer.
    1. Lance l'architecture RAG choisie
    2. Nettoie la règle générée
    3. Récupère description (depuis dataset) et référence (depuis dataset)
    4. Stocke dans last_response_state pour les boutons
    5. Retourne la règle prête à copier
    """
    if not query.strip():
        return 'Veuillez entrer une description de malware.'

    t0 = time.time()
    try:
        arch_func = RAG_FUNCTIONS.get(rag_type, rag_hybrid)
        rule_raw, docs = arch_func(query)
        latency = time.time() - t0

        # Nettoyer la règle
        rule_clean = clean_rule_output(rule_raw)

        # Référence — depuis le dataset (top doc récupéré)
        reference = docs[0].get('source_url', 'Non disponible') if docs else 'Non disponible'

        # Description dataset — depuis le dataset (top doc)
        dataset_desc = docs[0].get('description', '') if docs else ''

        # Stocker dans l'état (description générée par LLM au clic)
        last_response_state['rule']        = rule_clean
        last_response_state['query']       = query
        last_response_state['reference']   = reference
        last_response_state['dataset_desc']= dataset_desc
        last_response_state['docs']        = docs
        last_response_state['valid']       = is_valid_yara(rule_clean)
        last_response_state['latency']     = latency

        header = (
            f'Architecture : {rag_type} | LLM : {llm_choice}\n'
            f'Temps        : {latency:.2f}s | '
            f'Syntaxe valide : {"Oui ✓" if is_valid_yara(rule_clean) else "Non ✗"}\n'
            f'{"="*60}\n'
        )
        return header + rule_clean

    except Exception as e:
        return f'Erreur : {str(e)}'


def show_description() -> str:
    rule      = last_response_state.get('rule', '')
    query     = last_response_state.get('query', '')
    docs      = last_response_state.get('docs', [])
    dataset_d = last_response_state.get('dataset_desc', '')

    if not rule:
        return 'Générez une règle d\'abord.'

    print('[Description] Génération explication LLM...')
    llm_expl = explain_rule(rule, query)

    out  = f'DESCRIPTION & EXPLICATION\n{"="*50}\n'
    out += f'\n📋 Description dataset (règle source) :\n{dataset_d or "Non disponible"}\n'
    out += f'\n🤖 Explication ligne par ligne :\n{llm_expl}\n'

    # Extraire strings de la règle pour affichage
    strings = re.findall(r'(\$\w+\s*=\s*[^\n]+)', rule)
    if strings:
        out += f'\n📝 Strings détectées dans la règle :\n'
        for s in strings[:8]:
            out += f'   {s.strip()}\n'

    if docs:
        out += f'\n📚 Règles sources utilisées ({len(docs)}) :\n'
        for i, d in enumerate(docs[:3], 1):
            out += (
                f'  {i}. [{d.get("type","?")}] {d.get("rule_name","?")[:50]}\n'
                f'     Score   : {d.get("retrieval_score", d.get("rerank_score", "?"))}\n'
                f'     Dataset : {str(d.get("description",""))[:80]}...\n'
            )
    return out


def show_reference() -> str:
    """
    Appelé au clic sur le bouton Référence.
    Retourne les URLs des règles sources depuis le dataset.
    """
    reference = last_response_state.get('reference', '')
    docs      = last_response_state.get('docs', [])

    if not reference and not docs:
        return 'Générez une règle d\'abord.'

    out  = f'RÉFÉRENCES\n{"="*50}\n'
    out += f'\n🔗 Source principale :\n{reference or "Non disponible"}\n'

    if docs:
        out += f'\n📌 Sources des règles récupérées :\n'
        for i, d in enumerate(docs[:5], 1):
            url  = d.get('source_url', 'N/A')
            name = d.get('rule_name', 'N/A')[:50]
            typ  = d.get('type', '?')
            out += f'  {i}. [{typ}] {name}\n     {url}\n'
    return out


# ── Interface Gradio ──────────────────────────────────────────────────────
with gr.Blocks(title='YARA RAG System', theme=gr.themes.Soft()) as demo:

    gr.Markdown('# 🛡️ Système RAG — Génération de Règles YARA')
    gr.Markdown(
        'Décrivez un comportement malveillant → obtenez une **règle YARA complète** prête à copier.\n'
        '> **Dataset** : ~30K règles validées | **RAG** : Hybride, Rerank, Agentic'
    )

    with gr.Row():
        with gr.Column(scale=2):
            query_box = gr.Textbox(
                label='Description du malware',
                placeholder='Ex: Ransomware qui chiffre les fichiers avec AES et demande une rançon Bitcoin',
                lines=3
            )
        with gr.Column(scale=1):
            rag_selector = gr.Dropdown(
                choices=['RAG Hybride', 'RAG Rerank', 'Agentic RAG'],
                value='RAG Hybride',
                label='Architecture RAG'
            )
            llm_selector = gr.Dropdown(
                choices=[
                    'Phi-2 (2.7B)',
                    'TinyLlama (1.1B)',
                    'Mistral-7B'
                ],
                value='TinyLlama (1.1B)',
                label='LLM'
            )
            generate_btn = gr.Button('🚀 Générer la règle YARA', variant='primary', size='lg')

    rule_output = gr.Textbox(
        label='📄 Règle YARA générée (prête à copier)',
        lines=20

    )

    with gr.Row():
        desc_btn = gr.Button('📖 Description & Explication', variant='secondary')
        ref_btn  = gr.Button('🔗 Références',                variant='secondary')

    info_output = gr.Textbox(
        label='Informations',
        lines=12,
        interactive=False
    )

    gr.Examples(
        examples=[
            ['Ransomware encrypting files with AES requesting Bitcoin ransom',  'RAG Hybride',  'TinyLlama (1.1B)'],
            ['Keylogger capturing keystrokes with GetAsyncKeyState Windows API', 'RAG Rerank',   'TinyLlama (1.1B)'],
            ['Rootkit hiding processes by hooking SSDT kernel tables',           'Agentic RAG',  'TinyLlama (1.1B)'],
            ['Backdoor opening reverse shell on port 4444 using cmd.exe',        'RAG Hybride',  'TinyLlama (1.1B)'],
            ['Cryptominer using CPU to mine Monero XMR via stratum protocol',    'RAG Rerank',   'TinyLlama (1.1B)'],
        ],
        inputs=[query_box, rag_selector, llm_selector]
    )

    # Events
    generate_btn.click(
        fn=generate_rule,
        inputs=[query_box, rag_selector, llm_selector],
        outputs=[rule_output]
    )
    desc_btn.click(fn=show_description, inputs=[], outputs=[info_output])
    ref_btn.click( fn=show_reference,   inputs=[], outputs=[info_output])

print('Interface Gradio définie.')
demo.launch(share=True, server_name='0.0.0.0', server_port=7871)


Interface Gradio définie.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1714f5676a3fe1aa82.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
